# Neural Network Example

This notebook contains the same worked example that is embedded in `nn.qmd`.

This example shows how a small ReLU neural network can learn a sine curve from sample data. The goal is not speed. The goal is to make the connection between the notation and the code easy to follow.

As in the slides, we write the neural-network function as ${\rm ANN}(x)$. We write the training data as pairs $(x_i,y_i)$ and the mean squared error as
$$
\mathcal{L}
=
\frac{1}{N}\sum_{i=1}^{N}\left(y_i-{\rm ANN}(x_i)\right)^2.
$$

So the five steps are: define the neural network ${\rm ANN}(x)$, define the data $(x_i,y_i)$ and the loss $\mathcal{L}$, write one update for the weights and biases, repeat that update many times, and finally compare the learned curve with the reference curve.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

At the top, we collect a few parameters that you can vary without touching the rest of the example.

In [ ]:
num_data_points = 32
sine_frequency = 1.0
noise_level = 0.0
layer_widths = [1, 64, 64, 1]
num_training_steps = 5000

These five choices control the example:

- `num_data_points`: the number of training pairs $(x_i,y_i)$ sampled from the sine curve,
- `sine_frequency`: the frequency in the target curve `sin(sine_frequency * x)`,
- `noise_level`: the size of the random perturbation added to the clean sine values,
- `layer_widths`: the widths of the successive layers of the neural network,
- `num_training_steps`: the number of training updates performed.

For example, `layer_widths = [1, 64, 64, 1]` means one input variable, two hidden layers of width `64`, and one output value.

If you change `layer_widths`, keep the first and last entry equal to `1`, because both the input $x$ and the output $y$ are one-dimensional in this example.

Before we look at the network code, it helps to see the sample data the model is supposed to fit.

In [ ]:
def reference_curve(x):
    return jnp.sin(sine_frequency * jnp.asarray(x))


x_start, x_end = 0.0, 2.0 * jnp.pi
x_demo = jnp.linspace(x_start, x_end, num_data_points)
x_demo_true = reference_curve(x_demo)
y_demo = x_demo_true + noise_level * jax.random.normal(jax.random.PRNGKey(19), x_demo.shape)
x_demo_plot = jnp.linspace(x_start, x_end, 400)

fig, ax = plt.subplots(figsize=(5.6, 3.0))
ax.plot(
    x_demo_plot,
    reference_curve(x_demo_plot),
    color="black",
    linewidth=2.0,
    label="Reference Curve",
)
ax.scatter(x_demo, y_demo, color="#c44e52", s=24, zorder=4, label="Training Data")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title(f"Sample Data for the Sine Example (frequency = {sine_frequency:.2f})")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(frameon=True, fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

The task is simple to state: starting from the sample pairs $(x_i,y_i)$ on this curve, we want ${\rm ANN}(x)$ to recover the overall wave shape.

**Step 1. Define the neural network ${\rm ANN}(x)$.** Earlier in the section, the neural-network diagram showed the same basic rule over and over again: first apply a linear map, then apply the activation, then pass the result to the next layer. In symbols, one hidden layer takes an input vector $a$ and produces
$$
\sigma(aW+b),
$$
where $W$ is a weight matrix, $b$ is a bias vector, and $\sigma$ is the ReLU activation.

That is exactly what the code stores. Each layer is represented by:

- a weight matrix `weights`, which plays the role of $W$,
- a bias vector `bias`, which plays the role of $b$.

The hidden layers use **ReLU**, and the final layer stays linear. So the code below is just the slide notation written in JAX.

In [ ]:
def scale_inputs(inputs):
    return jnp.asarray(inputs) / jnp.pi - 1.0


def neural_network_predict(network_parameters, inputs):
    activations = scale_inputs(inputs)

    for layer_parameters in network_parameters[:-1]:
        activations = jax.nn.relu(
            activations @ layer_parameters["weights"] + layer_parameters["bias"]
        )

    output_layer = network_parameters[-1]
    return activations @ output_layer["weights"] + output_layer["bias"]

Here is how the code matches the notation:

- `scale_inputs(...)` rescales the input $x$ to a numerically convenient range,
- the loop over `network_parameters[:-1]` applies the hidden-layer rule
  $$
  a \mapsto \sigma(aW+b),
  $$
- the last two lines apply the final linear output layer.

So `neural_network_predict(...)` is the mathematical map

$$
x \mapsto {\rm ANN}(x)
$$

written in code.

The folded helper below creates the initial weights and biases.

In [ ]:
def initialize_neural_network_parameters(layer_widths, random_key):
    layer_keys = jax.random.split(random_key, len(layer_widths) - 1)
    network_parameters = []

    for layer_key, input_width, output_width in zip(
        layer_keys, layer_widths[:-1], layer_widths[1:]
    ):
        weight_key, bias_key = jax.random.split(layer_key)
        weight_scale = jnp.sqrt(2.0 / input_width)
        initial_bias_scale = 0.2
        network_parameters.append(
            {
                "weights": weight_scale
                * jax.random.normal(weight_key, (input_width, output_width)),
                "bias": initial_bias_scale
                * jax.random.normal(bias_key, (output_width,)),
            }
        )

    return tuple(network_parameters)

**Step 2. Define the sample data and the loss.** Next we create the training inputs and outputs, that is, the sample pairs $(x_i,y_i)$, and we define the **mean squared error**.

In [ ]:
x_start, x_end = 0.0, 2.0 * jnp.pi
training_inputs = jnp.linspace(x_start, x_end, num_data_points).reshape(-1, 1)
plot_inputs = jnp.linspace(x_start, x_end, 400).reshape(-1, 1)
clean_target_values = reference_curve(training_inputs[:, 0]).reshape(-1, 1)
clean_plot_values = reference_curve(plot_inputs[:, 0]).reshape(-1, 1)
noise_key = jax.random.PRNGKey(19)
target_values = clean_target_values + noise_level * jax.random.normal(
    noise_key, clean_target_values.shape
)


def mean_squared_error_loss(network_parameters, inputs, targets):
    predicted_values = neural_network_predict(network_parameters, inputs)
    return jnp.mean((predicted_values - targets) ** 2)

In this block, `training_inputs` stores the values $x_i$, and `target_values` stores the values $y_i$. The parameter `num_data_points` controls how many sample pairs we use, `sine_frequency` controls how quickly the target oscillates, and `noise_level` controls how strongly the data are perturbed away from the clean sine curve $y_i=\sin(\omega x_i)$.

The function `mean_squared_error_loss(...)` is therefore just the code form of $\mathcal{L}$: it evaluates ${\rm ANN}(x_i)$, compares it with $y_i$, squares the differences, and averages them.

**Step 3. Write one training update.** Now we tell JAX how to do one update of the weights and biases. Starting from the current values, we compute the loss, compute all derivatives, and then move the trainable numbers a little in the downhill direction.

In [ ]:
def take_training_step(network_parameters, inputs, targets, learning_rate):
    loss_value, loss_gradients = jax.value_and_grad(mean_squared_error_loss)(
        network_parameters, inputs, targets
    )

    updated_parameters = jax.tree_util.tree_map(
        lambda parameter, gradient: parameter - learning_rate * gradient,
        network_parameters,
        loss_gradients,
    )

    return updated_parameters, loss_value

The key line is the update

$$
W^{(j)} \leftarrow W^{(j)} - \eta \frac{\partial \mathcal{L}}{\partial W^{(j)}},
\qquad
b^{(j)} \leftarrow b^{(j)} - \eta \frac{\partial \mathcal{L}}{\partial b^{(j)}}
$$

for every layer $j$. Here `jax.value_and_grad(...)` is the bridge between mathematics and code: it returns both the scalar loss value $\mathcal{L}$ and the derivatives with respect to all weights and biases.

The input values still come from the same sine example on `[0,2\pi]` as in the earlier sections. We only rescale them internally inside the network so that the training process sees inputs on a friendlier numerical range.

The next folded block uses the architecture from `layer_widths`, initializes the parameter values, and chooses the learning rate $\eta$.

In [ ]:
network_parameters = initialize_neural_network_parameters(
    layer_widths, jax.random.PRNGKey(11)
)
learning_rate = 5.0e-3

**Step 4. Repeat the update many times.** One update is not enough. Training means repeating the same weight-and-bias update again and again.

In [ ]:
for step in range(num_training_steps):
    network_parameters, loss_value = take_training_step(
        network_parameters,
        training_inputs,
        target_values,
        learning_rate,
    )

    if (step + 1) % 500 == 0:
        print(f"step {step + 1:4d}: loss = {float(loss_value):.4e}")

trained_predictions = neural_network_predict(network_parameters, plot_inputs)

The printed **loss values** let us check whether training is actually driving $\mathcal{L}$ downward.

**Step 5. Compare the trained model with the reference curve.** After training, we compare the final function ${\rm ANN}(x)$ with both the clean sine curve and the chosen training data.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2))

ax.plot(
    plot_inputs[:, 0],
    clean_plot_values[:, 0],
    color="black",
    linewidth=2.0,
    label="Reference Curve",
)
ax.scatter(
    training_inputs[:, 0],
    target_values[:, 0],
    color="#c44e52",
    s=22,
    zorder=4,
    label=f"Training Data (noise = {noise_level:.2f})",
)
ax.plot(
    plot_inputs[:, 0],
    trained_predictions[:, 0],
    color="#1f77b4",
    linewidth=2.1,
    label="Neural Network Prediction",
)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title(
    f"Reference Curve versus Neural Network Prediction "
    f"(frequency = {sine_frequency:.2f}, noise = {noise_level:.2f})"
)
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(frameon=True, fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

So, in the notation of the slides, training always follows the same pattern:

- start with initial weights and biases,
- evaluate the prediction ${\rm ANN}(x_i)$,
- measure the error through $\mathcal{L}$,
- compute how each weight and bias should change,
- update the trainable numbers a little,
- and repeat.

That is the core idea behind neural-network training in JAX. More advanced tools such as Optax mainly package these same steps in a more efficient and convenient way.

**Further Tests**

- Reduce the data quality by using fewer training points and/or a larger `noise_level`.
- Change `sine_frequency` to make the target oscillate more slowly or more quickly on the same interval.
- Vary the network architecture: for instance, test a small model such as `layer_widths = [1, 20, 1]`, then compare it with a larger model that has more layers and more neurons per layer.
- Vary `num_training_steps`, from only a few iterations up to many thousands, and observe how the prediction changes as training is stopped earlier or allowed to continue longer.
